In [31]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

In [32]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-opus-4-6",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

This code defines a function `read_code_file(file_path)` that reads the contents of a file specified by `file_path` and returns it as plain text. It demonstrates how to open a file, read its contents, and return the result. Example usage is provided in the comments.

In [33]:
def read_code_file(file_input=None):
    """Reads code from a file and returns it as plain text. Accepts a file path string or Gradio file object."""
    if file_input is None:
        raise ValueError("Uploaded file must have a valid filename with extension.")
    # Handle Gradio file object (file-like with .read() method)
    if hasattr(file_input, 'read'):
        return file_input.read().decode('utf-8')
    # Handle file path string (e.g., from Gradio with type="filepath")
    if isinstance(file_input, str):
        with open(file_input, 'r') as f:
            return f.read()
    raise ValueError("file_input must be a path string or file-like object.")

# Used by the Gradio interface below - upload a file there to generate unit tests.

In [34]:
system_prompt = """
You are an expert software developer and a meticulous Quality Assurance (QA) engineer. Your task is to analyze a given code snippet or function, understand its logic, and generate a comprehensive suite of unit tests for it.

**Analyze the code for:**
*   Purpose and behavior
*   Input/Output formats and data types
*   Edge cases and potential failure points (e.g., invalid input, zero values, empty collections, exceptions)
*   Dependencies and required mocking strategies

**Generate unit tests with the following specifications:**
*   Use the `[Specify Testing Framework, e.g., pytest, JUnit, Jest]` framework.
*   Adhere to the `[Specify Principle, e.g., Arrange-Act-Assert (AAA)]` principle for structuring each test.
*   Ensure tests are independent and repeatable.
*   Provide clear and descriptive test names that explain the expected behavior (e.g., `should_return_zero_for_empty_list`, `test_throws_error_for_invalid_input`).
*   Include comments where necessary to explain complex logic or assumptions.
*   For dependencies, use `[Specify Mocking Library, e.g., Mock, Moq, Mockito]` for mocking.
*   Cover normal cases, edge cases, and exception handling scenarios.
*   Aim for high code coverage (line, branch, and method coverage).

**Output Format:**
*   Enclose the generated unit test code within distinct delimiters, for example:
    ```
    ###TEST_START###
    [Your generated test code here]
    ###TEST_END###
    ```
*   Do not include any additional natural language explanations or introductory text outside the specified delimiters in your final output, only the code and necessary comments within.
"""
# For future reference
# **Examples (Optional - include if using few-shot prompting):**
# [You can insert one or more examples of input code and the corresponding ideal unit test output here to guide the model further]


def user_prompt(code_text):
    return f"""Here is the code to analyze and generate unit tests for:
    ```
    {code_text}
    ```"""

# messages = [
#     {"role": "system", "content": system_prompt},
#     {"role": "user", "content": user_prompt(code_text)}
# ]

This code defines the `call_openai` function, which sends a chat completion request to the OpenAI API using a specified model, message history, and temperature. It initializes the OpenAI client with endpoint and API key from `MODEL_MAP`, sends the request, and returns the generated response content.

In [35]:
def call_openai(model, messages, temperature=0.7):
    client = OpenAI(
        base_url=MODEL_MAP[model]["endpoint"],
        api_key=MODEL_MAP[model]["api_key"]
    )
    response = client.chat.completions.create(
        model=MODEL_MAP[model]["model"],
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

### Gradio Interface for Code Unit Test Generation
This cell creates a Gradio interface that allows users to upload a code file and select a model for inference. The interface reads the uploaded file, generates the appropriate prompt, sends it to the selected model using the `call_openai` function, and displays the generated unit tests.

In [36]:
import gradio as gr

def get_model_options():
    return list(MODEL_MAP.keys())

def gradio_generate_tests(file_obj, model, temperature=0.7):
    if file_obj is None:
        return "Please upload a code file."
    code_text = read_code_file(file_obj)
    prompt = user_prompt(code_text)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    try:
        result = call_openai(model, messages, temperature)
    except Exception as e:
        return f"Error: {e}"
    return result

with gr.Blocks() as demo:
    gr.Markdown("## Upload a code file and select a model to generate unit tests")
    with gr.Row():
        file_input = gr.File(label="Upload code file (.py)")
        model_input = gr.Dropdown(choices=get_model_options(), value=get_model_options()[0], label="Select Model")
        temp_input = gr.Slider(minimum=0.0, maximum=1.0, value=0.7, step=0.05, label="Temperature")
    output = gr.Textbox(label="Generated Unit Tests", lines=20)
    run_btn = gr.Button("Generate Unit Tests")
    run_btn.click(fn=gradio_generate_tests, inputs=[file_input, model_input, temp_input], outputs=output)

demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/protocols/http/httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/fastapi/applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.12/site-packages/starlette/applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/home/thomas/Desktop/llm_engineering/.venv/lib/python3.